# Embedding-space validation of NRTK perturbations

**Scaffold notebook** — this is a runnable skeleton that documents the recommended pattern
for validating NRTK perturbers in feature-embedding space. The goal is to confirm that a
perturber's output is *monotonically* displaced from the original in a pretrained model's
embedding space as a function of its severity knob.

## What this notebook does

1. Loads a pretrained image model (default: a `torchvision` ResNet-50).
2. For each of several NRTK perturbers (photometric, geometric, optical), sweeps a severity
   parameter over a monotone grid.
3. For every image × severity, extracts the image embedding.
4. Plots cosine distance between the original and perturbed embeddings vs. severity.
5. Reports the Spearman rank correlation; a strictly monotone perturber should produce
   a ρ close to 1.0.

## What this notebook does *not* do (yet)

* Download a large image corpus. A small, bundled sample or a user-supplied directory
  is expected.
* Fine-tune or retrain the model.
* Guarantee any specific numeric threshold — the sweep output is diagnostic.

## Dependencies

```bash
pip install torch torchvision matplotlib scipy
pip install 'nrtk[graphics,pybsm]'
```

## Configuration

Edit the paths and knobs below. Everything downstream reads from `CONFIG`.

In [ ]:
from pathlib import Path

CONFIG = {
    "image_dir": Path("./sample_images"),
    "model": "resnet50",
    "num_severity_steps": 7,
    "max_images": 8,
    "device": "cpu",
    "seed": 0,
}
CONFIG

## Imports and image loading

When `CONFIG["image_dir"]` does not exist, the notebook synthesizes gradient tiles so it
still runs end-to-end in an unconfigured environment.

In [ ]:
from collections.abc import Callable, Iterable

import matplotlib.pyplot as plt
import numpy as np

from nrtk.interfaces import PerturbImage


def _load_images_from_dir(image_dir: Path, max_images: int) -> list[np.ndarray]:
    """Load up to ``max_images`` RGB images from ``image_dir``."""
    from PIL import Image

    suffixes = {".jpg", ".jpeg", ".png"}
    paths = [p for p in sorted(image_dir.iterdir()) if p.suffix.lower() in suffixes]
    return [np.asarray(Image.open(p).convert("RGB").resize((224, 224))) for p in paths[:max_images]]


def _synth_images(n: int, seed: int) -> list[np.ndarray]:
    """Synthesize ``n`` gradient/noise tiles for end-to-end scaffold runs."""
    rng = np.random.default_rng(seed)
    out: list[np.ndarray] = []
    base = np.linspace(0, 255, 224, dtype=np.uint8)
    tile = np.tile(base, (224, 1))
    for _ in range(n):
        rgb = np.stack([tile, np.flipud(tile), rng.integers(0, 256, (224, 224))], axis=-1)
        out.append(rgb.astype(np.uint8))
    return out


def load_images(image_dir: Path, max_images: int, seed: int) -> list[np.ndarray]:
    """Return a small list of images from disk, or synthesize them as a fallback."""
    if image_dir.exists():
        found = _load_images_from_dir(image_dir, max_images)
        if found:
            return found
    return _synth_images(max_images, seed)

## Load the embedding model

`embed(image)` takes an `(H, W, 3)` uint8 array and returns a 1-D normalized feature vector.

In [ ]:
def build_embedding_fn(model_name: str, device: str) -> Callable[[np.ndarray], np.ndarray]:
    """Return a unit-norm embedding function for ``model_name`` on ``device``."""
    if model_name != "resnet50":
        msg = f"Unknown model: {model_name!r}. Supported: 'resnet50'."
        raise ValueError(msg)
    import torch
    from torchvision import models, transforms

    weights = models.ResNet50_Weights.IMAGENET1K_V2
    model = models.resnet50(weights=weights).eval().to(device)
    model.fc = torch.nn.Identity()
    tfm = transforms.Compose(
        [
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ],
    )

    @torch.no_grad()
    def _embed(img: np.ndarray) -> np.ndarray:
        x = tfm(img).unsqueeze(0).to(device)
        v = model(x).cpu().numpy().ravel()
        return v / (np.linalg.norm(v) + 1e-12)

    return _embed

## Define the perturber sweeps

Each entry is `(name, factory)` where `factory(severity: float) -> PerturbImage`.
Severity 0.0 is (approximately) a no-op; 1.0 is the most severe configuration still
considered useful.

In [ ]:
def _photometric_sweeps() -> list[tuple[str, Callable[[float], PerturbImage]]]:
    """Return always-available photometric perturber sweeps."""
    from nrtk.impls.perturb_image.photometric._enhance.brightness_perturber import (
        BrightnessPerturber,
    )
    from nrtk.impls.perturb_image.photometric._noise.gaussian_noise_perturber import (
        GaussianNoisePerturber,
    )

    return [
        (
            "photometric:gaussian_noise",
            lambda s: GaussianNoisePerturber(mean=0.0, var=0.0001 + 0.05 * s, seed=0),
        ),
        (
            "photometric:brightness",
            lambda s: BrightnessPerturber(factor=1.0 + 0.9 * s),
        ),
    ]


def _geometric_sweep() -> tuple[str, Callable[[float], PerturbImage]] | None:
    """Return a geometric sweep entry when graphics/headless extras are installed."""
    try:
        from nrtk.impls.perturb_image.geometric.random import (
            RandomCropPerturber,
        )
    except ImportError as exc:
        print(f"[skip] geometric sweep unavailable: {exc}")
        return None

    def _crop_factory(severity: float) -> PerturbImage:
        shrink = max(int(224 * (1.0 - 0.5 * severity)), 32)
        return RandomCropPerturber(crop_size=(shrink, shrink), seed=0)

    return "geometric:random_crop", _crop_factory


def _optical_sweep() -> tuple[str, Callable[[float], PerturbImage]] | None:
    """Return an optical sweep entry when the pybsm extra is installed."""
    try:
        from nrtk.impls.perturb_image.optical._pybsm_presets import maritime_perturber
    except ImportError as exc:
        print(f"[skip] optical sweep unavailable: {exc}")
        return None

    def _factory(severity: float) -> PerturbImage:
        ihaze = 1 + int(round(severity * 2))
        return maritime_perturber(seed=0, is_static=True, ihaze=ihaze)

    return "optical:pybsm_maritime_ihaze", _factory


def build_perturber_sweeps() -> list[tuple[str, Callable[[float], PerturbImage]]]:
    """Collect every perturber sweep available in the current install."""
    sweeps = _photometric_sweeps()
    for optional in (_geometric_sweep(), _optical_sweep()):
        if optional is not None:
            sweeps.append(optional)
    return sweeps

## Run the sweep

For each perturber, for each severity, for each image, compute
`cosine_distance(embed(original), embed(perturbed))`. Plot mean ± std across images.

In [ ]:
def cosine_distance(a: np.ndarray, b: np.ndarray) -> float:
    """Cosine distance ``1 - <a, b> / (|a| |b|)``."""
    num = float(np.dot(a, b))
    den = float(np.linalg.norm(a) * np.linalg.norm(b) + 1e-12)
    return 1.0 - num / den


def sweep_perturber(
    factory: Callable[[float], PerturbImage],
    images: Iterable[np.ndarray],
    embed: Callable[[np.ndarray], np.ndarray],
    severities: np.ndarray,
) -> np.ndarray:
    """Return a ``(len(severities), len(images))`` matrix of cosine distances."""
    images_list = list(images)
    original_vecs = [embed(img) for img in images_list]
    distances = np.zeros((len(severities), len(images_list)))
    for i, s in enumerate(severities):
        perturber = factory(float(s))
        for j, (img, v0) in enumerate(zip(images_list, original_vecs, strict=True)):
            perturbed, _ = perturber.perturb(image=img)
            perturbed_u8 = perturbed if perturbed.dtype == np.uint8 else perturbed.astype(np.uint8)
            distances[i, j] = cosine_distance(v0, embed(perturbed_u8))
    return distances


def run_all_sweeps() -> tuple[dict[str, np.ndarray], np.ndarray]:
    """Execute every registered sweep and return ``(results, severities)``."""
    images = load_images(CONFIG["image_dir"], CONFIG["max_images"], CONFIG["seed"])
    embed = build_embedding_fn(CONFIG["model"], CONFIG["device"])
    severities = np.linspace(0.0, 1.0, CONFIG["num_severity_steps"])
    results: dict[str, np.ndarray] = {}
    for name, factory in build_perturber_sweeps():
        print(f"== sweeping {name} ==")
        results[name] = sweep_perturber(factory, images, embed, severities)
    return results, severities


# Uncomment to execute:
# results, severities = run_all_sweeps()

## Plot and diagnose monotonicity

A perturber whose embedding distance increases monotonically with severity is a good
sign that it is producing a coherent, controlled degradation. Non-monotone curves
(e.g., distance saturating or dropping at high severity) indicate either clipping
artifacts or a perturber whose high-severity setting produces a qualitatively different
image (e.g., a black frame) that the model maps far from the perturbation manifold.

In [ ]:
def plot_sweep_results(results: dict[str, np.ndarray], severities: np.ndarray) -> None:
    """Plot mean ± std cosine distance vs. severity for every sweep."""
    from scipy.stats import spearmanr

    _, ax = plt.subplots(figsize=(7, 4))
    for name, d in results.items():
        mean = d.mean(axis=1)
        std = d.std(axis=1)
        rho, _ = spearmanr(severities, mean)
        label = f"{name} (ρ={rho:.2f})"
        ax.errorbar(severities, mean, yerr=std, marker="o", capsize=3, label=label)
    ax.set_xlabel("Normalized severity")
    ax.set_ylabel("Cosine distance from original embedding")
    ax.set_title("Embedding distance vs. perturbation severity")
    ax.legend(fontsize=8)
    ax.grid(visible=True, linestyle="--", alpha=0.3)


# Uncomment once run_all_sweeps() has been executed:
# plot_sweep_results(results, severities)

## Extending this notebook

* Add a CLIP backbone in `build_embedding_fn` for text-image alignment checks.
* Swap the cosine distance metric for a calibrated perceptual distance (e.g. LPIPS).
* Plug in a real dataset by pointing `CONFIG["image_dir"]` at a directory of representative
  imagery for your deployment domain.
* Add per-class breakdowns by keeping labels alongside images and computing
  distance separately within each class.